# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)
df.head()

,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [7]:
low_claim = df["Total Claim Amount"] > 1000
response_yes= df["Response"] == "Yes"
# Create a new DataFrame with the filtered data

df_low_claim_response_yes = df[low_claim & response_yes]
# Display the first few rows and the shape of the new DataFrame

df_low_claim_response_yes.head()
df_low_claim_response_yes.shape
# The new DataFrame has 67 rows and 26 columns.

(67, 26)

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [6]:
df_yes = df[df["Response"] == "Yes"]

pivot_yes = (
    df_yes.pivot_table(
        index="Policy Type",
        columns="Gender",
        values=["Monthly Premium Auto", "Customer Lifetime Value"],
        aggfunc="mean"
    ).round(2)
)
pivot_yes
# The pivot table shows the average Monthly Premium Auto and Customer Lifetime Value for each Policy Type, broken

Customer Lifetime Value          Monthly Premium Auto       
Gender                               F        M                    F      M
Policy Type                                                                
Corporate Auto                 7712.63  7944.47                94.30  92.19
Personal Auto                  8339.79  7448.38                99.00  91.09
Special Auto                   7691.58  8247.09                92.31  86.34

In [8]:
pivot_claim_yes = (
    df_yes.pivot_table(
        index="Policy Type",
        columns="Gender",
        values="Total Claim Amount",
        aggfunc="mean"
    ).round(2)
)
pivot_claim_yes
# The pivot table shows the average Total Claim Amount for each Policy Type, broken down by Gender.

Gender,F,M
Policy Type,,
Corporate Auto,433.74,408.58
Personal Auto,452.97,457.01
Special Auto,453.28,429.53


## For customers who responded yes, the highest averages of total claims is for males with personal auto as policy type. The lowest goes to the policy type corporate auto with males. 

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [11]:
customer_per_state = df["State"].value_counts()
customer_per_state


state_over_500 = customer_per_state[customer_per_state > 500]
state_over_500


State
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
Name: count, dtype: int64

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [24]:
clv_value = (
    df.groupby(["Education", "Gender"])["Customer Lifetime Value"]
    .agg(["min", "median", "max"])
    .round(2)
    .reset_index()
)
clv_value

,Education,Gender,min,median,max
0,Bachelor,F,1904.00,5640.51,73225.96
1,Bachelor,M,1898.01,5548.03,67907.27
2,College,F,1898.68,5623.61,61850.19
3,College,M,1918.12,6005.85,61134.68
4,Doctor,F,2395.57,5332.46,44856.11
5,Doctor,M,2267.60,5577.67,32677.34
6,High School or Below,F,2144.92,6039.55,55277.45
7,High School or Below,M,1940.98,6286.73,83325.38
8,Master,F,2417.78,5729.86,51016.07
9,Master,M,2272.31,5579.10,50568.26


## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [26]:
df.columns


Index(['Unnamed: 0', 'Customer', 'State', 'Customer Lifetime Value',
       'Response', 'Coverage', 'Education', 'Effective To Date',
       'EmploymentStatus', 'Gender', 'Income', 'Location Code',
       'Marital Status', 'Monthly Premium Auto', 'Months Since Last Claim',
       'Months Since Policy Inception', 'Number of Open Complaints',
       'Number of Policies', 'Policy Type', 'Policy', 'Renew Offer Type',
       'Sales Channel', 'Total Claim Amount', 'Vehicle Class', 'Vehicle Size',
       'Vehicle Type'],
      dtype='object')

In [32]:
df["Effective To Date"] = pd.to_datetime(df["Effective To Date"])
df["Month"] = df["Effective To Date"].dt.month_name()

sales_state_month = (
    df.pivot_table(
        index="State",
        columns="Month",
        values="Policy",    
        aggfunc="count",
        fill_value=0
    )
)

month_order = [
    "January","February","March","April","May","June",
    "July","August","September","October","November","December"
]
sales_state_month = sales_state_month.reindex(columns=month_order, fill_value=0)

sales_state_month
# The pivot table shows the number of policies sold in each state for each month.

Month,January,February,March,April,May,June,July,August,September,October,November,December
State,,,,,,,,,,,,
Arizona,1008,929,0,0,0,0,0,0,0,0,0,0
California,1918,1634,0,0,0,0,0,0,0,0,0,0
Nevada,551,442,0,0,0,0,0,0,0,0,0,0
Oregon,1565,1344,0,0,0,0,0,0,0,0,0,0
Washington,463,425,0,0,0,0,0,0,0,0,0,0


6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [33]:
top3_states = (
    df["State"]
    .value_counts()
    .head(3)
    .index
)

top3_monthly = df[df["State"].isin(top3_states)].pivot_table(
    index="State",
    columns="Month",
    values="Policy",
    aggfunc="count",
    fill_value=0
).reindex(columns=month_order, fill_value=0)
top3_monthly
# The pivot table shows the number of policies sold in the top 3 states for each month

Month,January,February,March,April,May,June,July,August,September,October,November,December
State,,,,,,,,,,,,
Arizona,1008,929,0,0,0,0,0,0,0,0,0,0
California,1918,1634,0,0,0,0,0,0,0,0,0,0
Oregon,1565,1344,0,0,0,0,0,0,0,0,0,0


7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [37]:
long_df = df.melt(
    id_vars=["Sales Channel"],
    value_vars=["Response"],
    var_name="metric",
    value_name="value"
)

response_rate_long = (
    long_df.assign(value=lambda x: x["value"].str.lower())
           .groupby("Sales Channel")["value"]
           .apply(lambda s: (s == "yes").mean())
           .sort_values(ascending=False)
           .round(2)
           .reset_index(name="response_rate")
)

response_rate_long


,Sales Channel,response_rate
0,Agent,0.18
1,Web,0.11
2,Branch,0.11
3,Call Center,0.10
